# 어떤 고객이 통신 서비스를 해지하는가

## 1. 분석 개요

### 분석 주제

공개 데이터인 **Telco Customer Churn 데이터셋**을 이용하여 통신사 고객의 가입 기간, 계약 방식, 이용 서비스, 요금 특성과 서비스 이탈의 관계를 분석한다. 분석의 목적은 이탈과 관련된 주요 특성을 설명하고, 이탈 가능성이 높은 고객을 사전에 식별하여 고객 유지 활동의 우선순위를 정하는 데 활용할 수 있는 분류 모형을 구축하는 것이다.

이를 위해 데이터 품질 점검, 탐색적 분석, 전처리, 모형 학습과 검증, 성능 비교, 변수 영향 해석을 순서대로 수행한다. 단순 집단 간 이탈률과 여러 조건을 함께 고려한 결과를 구분하여 해석하고, 최종 모형은 이탈 고객 탐지 성능과 유지 활동의 효율성을 함께 고려해 선정한다.

이 분석에서 `Churn`은 고객의 통신 서비스 해지 여부를 나타낸다. `Yes`는 서비스를 해지한 고객(이탈 고객), `No`는 서비스를 유지한 고객을 뜻한다.

### 분석 질문

- **Q1.** 고객의 가입 기간, 계약 유형, 인터넷 서비스, 부가서비스, 결제 방식 및 요금 특성 가운데 **이탈률 차이가 크게 나타나는 요인**은 무엇이며, 여러 변수를 함께 고려한 뒤에도 이탈 가능성과 관련되는 요인은 무엇인가?
- **Q2.** 계약 유형과 가입 기간을 함께 고려했을 때 **이탈 위험이 집중되는 고객군은 누구**이며, **인터넷 서비스 및 핵심 부가서비스 이용 여부에 따라 그 위험은 어떻게 달라지는가?**
- **Q3.** 후보 분류 모형 가운데 이탈 고객에 대한 재현율을 우선하면서도 **정밀도와 PR·AUC의 균형이 가장 좋은 모형과 분류 기준값**은 무엇인가?

**종속변수:** `Churn` - 고객의 서비스 이탈 여부(`Yes`, `No`)를 나타내는 이진 범주형 변수

**설명변수:** `gender`, `SeniorCitizen`, `Partner`, `Dependents`, `tenure`, `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`, `Contract`, `PaperlessBilling`, `PaymentMethod`, `MonthlyCharges`, `TotalCharges`

`customerID`는 고객을 구분하기 위한 식별자이므로 예측을 위한 설명변수에서 제외한다.

### 이번 노트북에서 적용할 기법 - 머신러닝 분류
먼저 데이터의 구조, 중복, 결측값, 자료형과 범주 분포를 점검하고, 고객 특성별 이탈률과 수치형 분포를 탐색한다. 이후 자료를 계층화하여 학습 자료와 시험 자료로 분리하고, 결측값 처리·범주형 변수 변환·수치형 변수 조정은 학습 과정 안에서 수행하여 정보 누출을 방지한다.

단순 기준 모형과 함께 로지스틱 회귀, 랜덤포레스트, 그래디언트 부스팅 계열 모형을 비교한다. 후보 모형의 설정값은 학습 자료 내부의 교차검증으로 조정하며, 시험 자료는 최초 성능 평가에 한 번만 사용한다. 클래스 불균형 처리의 필요성은 학습 자료의 분포와 교차검증 결과를 바탕으로 판단하고, 재표본화를 사용할 경우에도 교차검증 과정 안에서만 적용한다.

이탈 고객을 유지 고객으로 잘못 예측하면 사전 대응 기회를 잃을 수 있으므로 **이탈 고객 재현율**을 우선 평가한다. 다만 모든 고객을 유지 활동 대상으로 선정하는 비효율을 막기 위해 **정밀도, F1점수, PR-AUC, ROC-AUC와 혼동행렬**도 함께 비교한다. 최종 모형과 분류 기준값은 이탈 탐지 성능뿐 아니라 실제로 접촉해야 할 고객 규모와 오탐 가능성을 함께 고려해 선택한다.

모형 해석 단계에서는 로지스틱 회귀의 계수 및 오즈비와 비선형 모형의 변수 영향도를 함께 살펴본다. 이때 변수 중요도를 원인이나 정책 효과로 해석하지 않고, 예측에 기여한 관련성으로 제한하여 설명한다.

> 통계적 유의성 판단 기준: 본 분석에서는 유의수준 (α = 0.05)를 기준으로 한다. 
> 따라서 p-value < 0.05이면 귀무가설을 기각하고 통계적으로 유의한 차이 또는 관계가 있다고 판단하며, p-value ≥ 0.05이면 귀무가설을 기각하지 않는다.

### 1-1. 분석 환경 설정

> 분석에 필요한 개별 함수 **import**

In [1]:
import sys

portfolio_path = r"C:\Data-Analysis-Portfolio"

if portfolio_path not in sys.path:
    sys.path.insert(0, portfolio_path)

from jihyelab import upload_data

In [2]:
from pandas import DataFrame, read_csv
from helpers import *
from IPython.display import display, Markdown

import os
import pandas as pd
import numpy as np
import statsmodels.api as sm

---
## 2. 데이터 소개

### 2-1. 데이터 로딩

In [4]:
# 캐글데이터 주소 모음집
churn = upload_data(
    "churn",
    local=r"C:\Data-Analysis-Portfolio\dataset_repository",
)

churn.head()

📚 통신사 고객의 인구통계·가입 기간·계약 방식·이용 서비스·요금 정보를 바탕으로 고객의 서비스 이탈 여부를 분석하고 예측하는 분류 데이터셋 (출처: https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

    field             description
--  ----------------  ---------------------------------------------------
 0  customerID        고객 식별번호
 1  gender            고객 성별 (Female, Male)
 2  SeniorCitizen     고령자 여부 (고령자=1, 비고령자=0)
 3  Partner           배우자 보유 여부 (Yes, No)
 4  Dependents        부양가족 보유 여부 (Yes, No)
 5  tenure            서비스 가입 기간(개월)
 6  PhoneService      전화 서비스 이용 여부 (Yes, No)
 7  MultipleLines     다중 전화회선 이용 여부 (Yes, No, No phone service)
 8  InternetService   인터넷 서비스 유형 (DSL, Fiber optic, No)
 9  OnlineSecurity    온라인 보안 서비스 이용 여부
10  OnlineBackup      온라인 백업 서비스 이용 여부
11  DeviceProtection  기기 보호 서비스 이용 여부
12  TechSupport       기술 지원 서비스 이용 여부
13  StreamingTV       TV 스트리밍 서비스 이용 여부
14  StreamingMovies   영화 스트리밍 서비스 이용 여부
15  Contract          계약 유형 (Month-to-month, One year, Two year)
16  PaperlessBilling  전자 청구서 이용 여부 (Yes,

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
churn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


#### 문자열 공백 제거
- 문자형 변수의 앞뒤 불필요한 공백을 제거한다.
    - 같은 값이 공백 때문에 서로 다른 범주로 인식되는 오류를 방지하기 위해 사용한다.

In [6]:
for col in churn.select_dtypes(include=['object', 'category']).columns:
    churn[col] = churn[col].astype(str).str.strip()

### 2-2. 출처와 분석 단위

- **출처:** Telco Customer Churn - Kaggle 배포본 [`blastchar/telco-customer-churn`](https://www.kaggle.com/datasets/blastchar/telco-customer-churn), IBM에서 제공한 통신사 고객 이탈 예제 데이터
- **수집 기간:** 배포 자료에 구체적인 수집 연도와 기간이 명시되어 있지 않음
- **분석 단위:** 1행 = 통신사 고객 1명의 인구통계 특성·가입 기간·계약 방식·이용 서비스·요금 정보와 해당 고객의 서비스 이탈 여부(`Churn`) 기록

### 2-3. 변수 정의서

| 변수명               | 의미                           | 분석상 역할         | 자료형       | 값·단위                                                                                                                                             |
| -------------------- | ------------------------------ | ------------------- | ------------ | ---------------------------------------------------------------------------------------------------------------------------------------------------- |
| `customerID`       | 고객별 고유 식별자             | 식별변수·모형 제외 | 문자열       | 고객별 고유값                                                                                                                                        |
| `gender`           | 고객의 성별                    | 설명변수            | 명목형       | `Female`=여성, `Male`=남성                                                                                                                       |
| `SeniorCitizen`    | 고령자 여부                    | 설명변수            | 이진 범주형  | `1`=고령자, `0`=비고령자                                                                                                                         |
| `Partner`          | 배우자 보유 여부               | 설명변수            | 이진 범주형  | `Yes`=있음, `No`=없음                                                                                                                            |
| `Dependents`       | 부양가족 보유 여부             | 설명변수            | 이진 범주형  | `Yes`=있음, `No`=없음                                                                                                                            |
| `tenure`           | 현재 통신사 가입 기간          | 설명변수            | 수치형·정수 | 개월                                                                                                                                                 |
| `PhoneService`     | 전화 서비스 이용 여부          | 설명변수            | 이진 범주형  | `Yes`=이용, `No`=미이용                                                                                                                          |
| `MultipleLines`    | 다중 전화회선 이용 상태        | 설명변수            | 명목형       | `Yes`, `No`, `No phone service`                                                                                                                |
| `InternetService`  | 인터넷 서비스 유형             | 설명변수            | 명목형       | `DSL`, `Fiber optic`, `No`                                                                                                                     |
| `OnlineSecurity`   | 온라인 보안 서비스 이용 상태   | 설명변수            | 명목형       | `Yes`, `No`, `No internet service`                                                                                                             |
| `OnlineBackup`     | 온라인 백업 서비스 이용 상태   | 설명변수            | 명목형       | `Yes`, `No`, `No internet service`                                                                                                             |
| `DeviceProtection` | 기기 보호 서비스 이용 상태     | 설명변수            | 명목형       | `Yes`, `No`, `No internet service`                                                                                                             |
| `TechSupport`      | 기술 지원 서비스 이용 상태     | 설명변수            | 명목형       | `Yes`, `No`, `No internet service`                                                                                                             |
| `StreamingTV`      | TV 스트리밍 서비스 이용 상태   | 설명변수            | 명목형       | `Yes`, `No`, `No internet service`                                                                                                             |
| `StreamingMovies`  | 영화 스트리밍 서비스 이용 상태 | 설명변수            | 명목형       | `Yes`, `No`, `No internet service`                                                                                                             |
| `Contract`         | 계약 유형                      | 설명변수            | 명목형       | `Month-to-month`=월 단위, `One year`=1년, `Two year`=2년                                                                                       |
| `PaperlessBilling` | 전자 청구서 이용 여부          | 설명변수            | 이진 범주형  | `Yes`=이용, `No`=미이용                                                                                                                          |
| `PaymentMethod`    | 요금 결제 방법                 | 설명변수            | 명목형       | `Electronic check`=전자수표, `Mailed check`=우편수표, `Bank transfer (automatic)`=은행 자동이체, `Credit card (automatic)`=신용카드 자동이체 |
| `MonthlyCharges`   | 고객에게 청구되는 월 이용 요금 | 설명변수            | 수치형·연속 | 원자료에 통화 단위 미표기                                                                                                                            |
| `TotalCharges`     | 가입 후 누적 청구 요금         | 설명변수            | 수치형·연속 | 원자료에 통화 단위 미표기                                                                                                                            |
| `Churn`            | 서비스 이탈 여부               | 종속변수            | 이진 범주형  | `Yes`=이탈, `No`=유지                                                                                                                            |

**데이터 및 전처리 원칙**

- 데이터는 7,043개 고객과 21개 변수로 구성되어 있으며, **`Churn`을 예측하는 이진 분류 문제로 정의한다.**
- `customerID`는 고유 식별자이므로 중복 고객 점검에만 사용하고 모형에서는 제외한다.
- `SeniorCitizen`은 숫자로 저장되어 있어도 크기와 간격에 의미가 없으므로 범주형으로 처리한다.
- `No phone service`와 `No internet service`는 누락된 값이 아니라 해당 상위 서비스를 이용하지 않는다는 정보이므로 유효한 범주로 유지한다.
- `TotalCharges`는 공백 문자열을 포함하므로 숫자형으로 변환한다. 공백 행이 가입 기간 0개월인 신규 고객인지 먼저 확인한 뒤, 그 의미에 따라 결측 처리 또는 제외 여부를 결정한다.
- 범주형 설명변수는 모형에 맞게 변환하고, 종속변수는 모델링 단계에서 `Yes=1`, `No=0`으로 부호화한다. 이 변환 규칙은 학습 자료에 적합한 전처리 과정 안에 포함한다.
- `TotalCharges`는 가입 기간과 월 요금이 누적된 값이므로 `tenure`, `MonthlyCharges`와의 상관 및 다중공선성을 점검한다. 설명 목적의 로지스틱 회귀에서는 변수 제외, 변환 또는 규제를 검토하고, 예측 모형에서는 교차검증 성능을 기준으로 포함 여부를 결정한다.

### 2-4. 초기 가설

- **H1.** 가입 기간이 짧고 월 단위 계약을 이용하는 고객은 가입 기간이 길거나 장기 계약을 이용하는 고객보다 이탈률이 높을 것이다. 여러 고객 특성을 함께 고려한 후에도 짧은 가입 기간과 월 단위 계약은 높은 이탈 가능성과 관련될 것이다.
- **H2.** 인터넷 이용 고객 가운데 온라인 보안 또는 기술 지원 서비스를 이용하지 않는 고객은 해당 서비스를 이용하는 고객보다 이탈률이 높을 것이다. 이 차이는 인터넷 서비스 유형과 월 요금을 함께 고려한 후에도 일부 유지될 것이다.
- **H3.** 고객 특성 사이의 비선형 관계와 상호작용을 반영할 수 있는 트리 기반 앙상블 모형은 로지스틱 회귀보다 PR-AUC가 높을 것이다. 다만 이탈 고객 재현율과 정밀도의 균형 및 해석 가능성까지 고려하면 최종 선택이 반드시 가장 복잡한 모형으로 결정되지는 않을 것이다.

위 가설은 분석 전에 설정한 예상이다. H1과 H2는 집단별 이탈률 차이, 효과 크기, 오즈비와 신뢰구간을 중심으로 평가하며, 통계적 유의성만으로 실무적 중요성을 판단하지 않는다. H3는 동일한 데이터 분할과 교차검증 조건에서 얻은 재현율, 정밀도, F1, PR-AUC 및 ROC-AUC를 기준으로 평가한다.

### 2-5. 데이터의 한계

- **관찰 자료이므로 인과관계를 확인할 수 없다.** 계약 유형, 서비스 이용 여부 및 요금과 이탈 사이의 관련성은 확인할 수 있지만, 특정 특성이 이탈을 직접 유발했다고 단정할 수 없다. 예를 들어 월 단위 계약은 이탈의 원인이라기보다 원래 이동 의향이 높은 고객이 선택한 결과일 수 있다.
- **이탈의 직접적인 이유와 고객 경험 정보가 없다.** 서비스 품질, 통신 장애, 고객센터 이용 경험, 경쟁사 제안, 할인 및 프로모션, 민원 기록 등이 없어 고객이 실제로 해지한 이유를 충분히 설명하기 어렵다.
- **시간에 따른 변화와 이탈 시점을 분석하기 어렵다.** 데이터에는 고객별 월간 이용량, 요금 변화, 서비스 추가·해지 과정과 같은 이력 정보가 없으므로 이탈 직전의 행동 변화를 포착할 수 없다. 또한 관찰 종료 시점이 명확하지 않아 생존분석 형태로 이탈 시점을 다루기 어렵다.
- **표본의 대표성과 다른 환경에서의 일반화 가능성이 불분명하다.** 자료의 수집 시점, 지역, 통신사 운영 정책과 표본 추출 과정이 충분히 제시되지 않아 분석 결과를 현재 시장이나 다른 통신사의 고객에게 그대로 적용하기 어렵다. 실제 적용 전에는 최신 자사 자료를 이용한 외부 검증과 성능 변화 점검이 필요하다.
- **높은 예측 성능이 이탈 방지 효과나 사업 수익을 보장하지 않는다.** 모형은 이탈 가능성이 높은 고객을 식별할 뿐, 어떤 고객이 유지 제안에 반응할지는 알려주지 않는다. 실제 운영에서는 유지 활동의 비용, 고객별 예상 가치, 접촉 가능 규모를 반영하고 대조 실험을 통해 개입의 순효과를 검증해야 한다.